# 19 · 客户端与 SDK：字节流如何成为可靠结果

CLI、SDK 与 TUI 都在投影同一个任务，但入口不同不能改变成功标准。子进程 exit=0、收到一条 JSON 或展示“完成”，都不足以替代核心验证结果。
本章分别练习 JSONL 帧边界、真实子进程退出、请求关联、局部取消与只读投影。代码是标准库传输实验，
没有启动真实 Codex CLI、Python SDK 或 TypeScript SDK。

**运行范围**：本章直接使用最终包中的共享组件，逐格验证本专题机制；不会启动模型、访问真实收费 API 或使用个人凭据。每次从第一格顺序运行，运行文件进入独立的 `runs/` 目录。改动参数后，从相应状态初始化格重新运行。


In [1]:
from pathlib import Path
import sys
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/'code/mini_codex/__init__.py').is_file())
sys.path.insert(0, str(ROOT/'code'))
from mini_codex.learning import load_example, show_source, fresh_run_dir, run_python, run_chapter
from unittest import TestCase
expect = TestCase()  # 只借用异常断言；下面逐格调用实现，不运行总测试替代实验。


## 源码事实与教学选择

固定对照提交：`c0cc9fc820f62e4dbdde68214dd6d792d6c8a7ee`。以下链接说明 Codex 对应机制；Python 类型和实验状态机是教学选择，运行它们不等于运行 Codex。

- [S19-12 · CodexExec.run](https://github.com/openai/codex/blob/c0cc9fc820f62e4dbdde68214dd6d792d6c8a7ee/sdk/typescript/src/exec.ts#L230-L264)：TS exec 消费完行流后检查 spawn 错误与退出码/信号；finally 关闭 readline 并尝试 kill 本子进程。
- [S19-17 · route_response / fail_all](https://github.com/openai/codex/blob/c0cc9fc820f62e4dbdde68214dd6d792d6c8a7ee/sdk/python/src/openai_codex/_message_router.py#L258-L280)：MessageRouter 按响应 ID 精确唤醒 waiter，未知 ID 忽略；fail_all 清空本 router waiter 并向等待者交付异常。
- [S19-24 · enqueue_thread_notification](https://github.com/openai/codex/blob/c0cc9fc820f62e4dbdde68214dd6d792d6c8a7ee/codex-rs/tui/src/app/thread_routing.rs#L1226-L1282)：TUI 按线程保存事件与 receiver：活跃线程通知进入显示队列，非活跃线程写入缓冲；widget 消费该线程事件。


## 实验 1 · 字节分块与 JSON 帧不是同一层


网络/管道可以在中文 UTF-8 字符中间切块。只有换行提交了完整帧才解码 JSON；EOF 时残留半帧必须报错。
把 CHUNK 改成 1、2、7，解析结果应相同。坏帧之前已经交付的合法前缀应保留。


In [2]:
client = load_example('19', 'protocol_lab')
import json
CHUNK = 2
payload = ({'kind':'turn.started','text':'检查任务'}, {'kind':'turn.completed','text':'完成'})
wire = b''.join(json.dumps(item,ensure_ascii=False).encode()+b'\n' for item in payload)
decoder = client.JsonlDecoder()
parsed = []
for pos in range(0,len(wire),CHUNK):
    parsed.extend(decoder.feed(wire[pos:pos+CHUNK]))
decoder.finish()
assert parsed == list(payload)
truncated = client.JsonlDecoder()
list(truncated.feed(b'{"kind":"partial"'))
with expect.assertRaisesRegex(client.ProtocolError, 'truncated_frame'):
    truncated.finish()
stream = client.JsonlDecoder().feed(b'{"ok":true}\nnot-json\n')
assert next(stream) == {'ok':True}
with expect.assertRaisesRegex(client.ProtocolError, 'invalid_frame'):
    next(stream)
with expect.assertRaisesRegex(client.ProtocolError, 'frame_too_large'):
    list(client.JsonlDecoder(max_frame_bytes=4).feed(b'12345'))
print({'chunk_bytes':CHUNK, 'decoded_events':parsed})


{'chunk_bytes': 2, 'decoded_events': [{'kind': 'turn.started', 'text': '检查任务'}, {'kind': 'turn.completed', 'text': '完成'}]}


## 实验 2 · 合法 stdout 不能掩盖子进程失败


启动一个真实自有 Python 子进程：它先输出合法 started 帧，再向 stderr 写诊断，最后退出 3。
SDK 必须同时检查事件与退出状态。stdout 是机器协议，stderr 是诊断，混读会破坏 JSONL。


In [3]:
import subprocess, sys
run_dir = fresh_run_dir('ch19')
script = run_dir/'failing_client.py'
script.write_text('import json,sys\nprint(json.dumps({"kind":"turn.started"}),flush=True)\nprint("fixture failure",file=sys.stderr)\nraise SystemExit(3)\n')
process = subprocess.run([sys.executable,str(script)],capture_output=True)
decoder = client.JsonlDecoder()
events = list(decoder.feed(process.stdout))
decoder.finish()
assert events == [{'kind':'turn.started'}]
assert process.returncode == 3 and b'fixture failure' in process.stderr
assert not any(e['kind']=='turn.completed' for e in events)
print({'exit_code':process.returncode,'events':events,'stderr':process.stderr.decode().strip()})


{'exit_code': 3, 'events': [{'kind': 'turn.started'}], 'stderr': 'fixture failure'}


## 实验 3 · 请求精确关联，旧连接不能回答新连接


两个 pending waiter 按 ID 各自取走一次；通知不应猜测唤醒哪个。重连使用新的 epoch，
旧 reader 即使收到相同数字 ID，也不能解决新 waiter。断连只报告远端状态 unknown，不伪造取消成功。


In [4]:
router = client.PendingRouter(epoch=1)
a, b = router.register(7), router.register(8)
assert router.response(1,8,{'owner':'B'})
assert b.result() == {'owner':'B'} and not a.done()
assert not router.response(1,8,{'duplicate':True})
router.disconnect()
with expect.assertRaisesRegex(ConnectionError,'remote_status_unknown'):
    a.result()
new = client.PendingRouter(epoch=2)
fresh = new.register(7)
assert not new.response(1,7,{'stale':True}) and not fresh.done()
assert new.response(2,7,{'fresh':True}) and fresh.result() == {'fresh':True}
local_wait = new.register(9)
assert new.cancel_wait(9) and local_wait.cancelled()
assert not new.response(2,9,{'late':True})
print('旧 epoch、重复响应与取消后的响应均未唤醒错误 waiter。')


旧 epoch、重复响应与取消后的响应均未唤醒错误 waiter。


## 实验 4 · 取消自有子进程，不杀其他调用


先让两个真实子进程分别写文件并发出 READY，再终止其中一个。另一个应仍能按自己的输入正常结束。
已写文件不会因 terminate 回滚。这验证本机自有进程的边界；对共享 daemon 或远端 turn，必须使用相应中断协议。


In [5]:
worker = run_dir/'owned_worker.py'
worker.write_text('from pathlib import Path\nimport sys\nPath(sys.argv[1]).write_text("effect committed")\nprint("READY",flush=True)\nsys.stdin.readline()\nprint("DONE",flush=True)\n')
owned = subprocess.Popen([sys.executable,str(worker),str(run_dir/'owned-effect.txt')],
    stdin=subprocess.PIPE,stdout=subprocess.PIPE,stderr=subprocess.PIPE,text=True)
sibling = subprocess.Popen([sys.executable,str(worker),str(run_dir/'sibling-effect.txt')],
    stdin=subprocess.PIPE,stdout=subprocess.PIPE,stderr=subprocess.PIPE,text=True)
try:
    assert owned.stdout.readline().strip() == 'READY'
    assert sibling.stdout.readline().strip() == 'READY'
    owned.terminate()
    owned.wait(timeout=3)
    assert owned.returncode < 0 and sibling.poll() is None
    assert (run_dir/'owned-effect.txt').read_text() == 'effect committed'
    stdout, stderr = sibling.communicate('finish\n',timeout=3)
    assert sibling.returncode == 0 and stdout.strip() == 'DONE'
    print({'cancelled_returncode':owned.returncode,'sibling_returncode':sibling.returncode,'effect_preserved':True})
finally:
    for child in (owned,sibling):
        if child.poll() is None:
            child.kill()
        child.communicate(timeout=3)


{'cancelled_returncode': -15, 'sibling_returncode': 0, 'effect_preserved': True}


## 实验 5 · 界面是只读投影，不能制造完成事实


Projection 只接收选中 thread 的递增事件；重复事件与其他线程事件被忽略。
它没有 Scheduler 引用，所以更新屏幕不会启动任务。该简化投影不处理缺失 seq 的补洞；换客户端恢复还需要服务端补读协议。


In [6]:
view = client.Projection('thread-A')
events = [
    {'thread_id':'thread-A','seq':1,'kind':'started','text':'checking'},
    {'thread_id':'thread-B','seq':2,'kind':'completed','text':'other task'},
    {'thread_id':'thread-A','seq':1,'kind':'started','text':'duplicate'},
    {'thread_id':'thread-A','seq':3,'kind':'failed','text':'check exited 3'}]
accepted = [view.accept(event) for event in events]
assert accepted == [True,False,False,True]
assert view.lines == ['started: checking','failed: check exited 3']
assert not any(line.startswith('completed:') for line in view.lines)
print({'accepted':accepted,'display':view.lines})


{'accepted': [True, False, False, True], 'display': ['started: checking', 'failed: check exited 3']}


## 你现在可以解释什么

能把传输完整性、进程退出、请求回执和业务成功分开判断；能解释取消一个 waiter 与取消任务的区别。本章没有完整 CLI 参数层、TS/Python SDK、真实 TTY 或核心验证器。

**验证边界**：本章断言验证当前局部实现与合成输入。未运行固定提交的 Rust 测试，未验证真实模型、外部服务或累计 Agent 的端到端行为。
